# OpenPlaque RCA Automatic Candidate Gallery

This notebook uses the clean `main`-derived branch and source CCTA **series 7**. It has **no sliders, no widgets, no click handling, and no manual coordinates**.

It automatically finds a small set of high-information axial slices for (1) the aortic-root/ostium region and (2) small contrast-filled coronary-like structures. It then renders numbered static galleries for visual review. This is a triage step, not an anatomical RCA classifier.

Use **Runtime → Run all**. Google Drive mounts first.


In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Fresh clone of the clean main-derived branch; standard OpenPlaque dependencies only.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC = Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from openplaque.study import OpenPlaqueStudy
from openplaque.source_candidates import find_informative_slices, find_root_informative_slices
print('OpenPlaque source:', SRC)


## Load source CCTA series 7
The DICOM ZIP is copied from Drive to local Colab storage before extraction/scanning.


In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_candidate_gallery'
SOURCE_SERIES = 7

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f'Missing {DRIVE_ZIP}')

t = time.time()
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying Full_DICOM.zip ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB) to local disk...', flush=True)
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
    print(f'Copy finished in {time.time()-t:.1f}s', flush=True)
else:
    print('Local ZIP already staged.')

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
print('Extracting/scanning DICOM locally...', flush=True)
t = time.time()
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
print(f'Scan finished in {time.time()-t:.1f}s; {len(study.series)} series found.', flush=True)
source_img, source, source_files = study.load_series(SOURCE_SERIES)
spacing = source_img.GetSpacing()
print('Loaded source series:', SOURCE_SERIES)
print('Shape zyx:', source.shape)
print('Spacing xyz mm:', spacing)


## Automatically rank the most informative slices
The score favors bright, small-caliber tubular structures and suppresses large central blood pools. The root search uses a broad automatically estimated aortic-root window.


In [ ]:
print('Finding aortic-root/ostium-informative slices...', flush=True)
t = time.time()
root_candidates, root_window = find_root_informative_slices(source, spacing, n=12, step=1, min_separation_mm=1.5)
print(f'Root search finished in {time.time()-t:.1f}s; window z={root_window[0]}..{root_window[1]-1}')

print('Finding general coronary-informative slices...', flush=True)
t = time.time()
cor_candidates = find_informative_slices(source, spacing, z_fraction=(0.42,0.82), n=12, step=2, min_separation_mm=2.5)
print(f'General coronary search finished in {time.time()-t:.1f}s')

def candidate_table(items, prefix):
    return pd.DataFrame([
        dict(candidate=f'{prefix}{i+1}', z=c.z, y=c.y, x=c.x, score=c.score,
             HU=c.hu, vesselness=c.vesselness, local_radius_mm=c.local_radius_mm)
        for i,c in enumerate(items)
    ])

root_df = candidate_table(root_candidates, 'O')
cor_df = candidate_table(cor_candidates, 'C')
print('\nOSTIUM/ROOT CANDIDATES')
display(root_df)
print('\nGENERAL CORONARY CANDIDATES')
display(cor_df)


## Static candidate galleries
Each panel is a real axial slice from series 7. The crosshair marks the highest-scoring small-vessel point found on that slice. Use the **panel label** (for example `O3` or `C7`) when referring to a candidate.


In [ ]:
OUTDIR = ROOT / 'RCA_Candidate_Gallery'
OUTDIR.mkdir(parents=True, exist_ok=True)

def show_gallery(items, prefix, title, filename, crop_radius=85):
    if not items:
        print('No candidates found for', title)
        return
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for i, ax in enumerate(axes.ravel()):
        if i >= len(items):
            ax.axis('off')
            continue
        c = items[i]
        z, y, x = c.z, c.y, c.x
        r = int(crop_radius)
        y0, y1 = max(0, y-r), min(source.shape[1], y+r+1)
        x0, x1 = max(0, x-r), min(source.shape[2], x+r+1)
        ax.imshow(source[z, y0:y1, x0:x1], cmap='gray', vmin=-200, vmax=800)
        ax.axvline(x-x0, linewidth=0.9)
        ax.axhline(y-y0, linewidth=0.9)
        ax.scatter([x-x0], [y-y0], s=45, facecolors='none')
        ax.set_title(
            f'{prefix}{i+1}: z={z}, x={x}, y={y}\n'
            f'HU={c.hu:.0f}, score={c.score:.3f}, r={c.local_radius_mm:.1f}mm'
        )
        ax.axis('off')
    fig.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0,0,1,0.96])
    save_path = OUTDIR / filename
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', save_path)

show_gallery(
    root_candidates, 'O',
    f'RCA review: most informative aortic-root / ostium slices (root window z={root_window[0]}..{root_window[1]-1})',
    'RCA_ostium_informative_gallery.png',
    crop_radius=95,
)

show_gallery(
    cor_candidates, 'C',
    'RCA review: most informative small-vessel / coronary slices',
    'RCA_coronary_informative_gallery.png',
    crop_radius=80,
)


## Context montage around the root window
This wider montage prevents a high vesselness score from being mistaken for the RCA simply because it looks tubular in a tight crop.


In [ ]:
z0, z1 = root_window
zs = np.linspace(z0, z1-1, 12).round().astype(int)
fig, axes = plt.subplots(3,4,figsize=(16,12))
for ax, z in zip(axes.ravel(), zs):
    ax.imshow(source[z], cmap='gray', vmin=-200, vmax=800)
    ax.set_title(f'root context z={z}')
    ax.axis('off')
plt.tight_layout()
context_path = OUTDIR / 'RCA_root_context_gallery.png'
fig.savefig(context_path, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved:', context_path)

print('\nCANDIDATE GALLERY COMPLETE.')
print('Upload the two candidate galleries (and root-context gallery if needed).')
print('No sliders or manual navigation are required.')
